# Data Quality Metrics

This notebook calculates reusable data quality metrics across Silver and Gold tables, including row counts, null business keys, duplicate grain records, and referential integrity issues.

In [0]:
from pyspark.sql import functions as F

In [0]:
SILVER_BASE_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist"
)

SILVER_TABLE_CONFIG = {
    "category_translation": {
        "business_keys": ["product_category_name"],
        "grain": ["product_category_name"],
    },
    "customers": {
        "business_keys": ["customer_id"],
        "grain": ["customer_id"],
    },
    "geolocation": {
    "business_keys": ["geolocation_zip_code_prefix"],
    "grain": None,
},
    "order_items": {
        "business_keys": ["order_id", "order_item_id"],
        "grain": ["order_id", "order_item_id"],
    },
    "order_payments": {
        "business_keys": ["order_id", "payment_sequential"],
        "grain": ["order_id", "payment_sequential"],
    },
    "order_reviews": {
    "business_keys": ["review_id", "order_id"],
    "grain": ["review_id", "order_id"],
},
    "orders": {
        "business_keys": ["order_id"],
        "grain": ["order_id"],
    },
    "products": {
        "business_keys": ["product_id"],
        "grain": ["product_id"],
    },
    "sellers": {
        "business_keys": ["seller_id"],
        "grain": ["seller_id"],
    },
}

In [0]:
silver_tables = {}

for table_name in SILVER_TABLE_CONFIG:
    silver_tables[table_name] = (
        spark.read
        .format("delta")
        .load(f"{SILVER_BASE_PATH}/{table_name}")
    )

print(f"Loaded {len(silver_tables)} Silver tables.")

In [0]:
for table_name, config in SILVER_TABLE_CONFIG.items():
    table_columns = set(silver_tables[table_name].columns)

    required_columns = set(config["business_keys"])

    if config["grain"]:
        required_columns.update(config["grain"])

    missing_columns = required_columns - table_columns

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

print("Silver metric configuration validation passed.")

In [0]:
silver_metric_rows = []

for table_name, config in SILVER_TABLE_CONFIG.items():
    table_df = silver_tables[table_name]

    row_count = table_df.count()

    null_key_condition = None

    for key_column in config["business_keys"]:
        current_condition = F.col(key_column).isNull()

        null_key_condition = (
            current_condition
            if null_key_condition is None
            else null_key_condition | current_condition
        )

    null_business_key_count = (
        table_df
        .filter(null_key_condition)
        .count()
    )

    if config["grain"]:
        duplicate_grain_count = (
            table_df
            .groupBy(*config["grain"])
            .count()
            .filter(F.col("count") > 1)
            .agg(
                F.coalesce(
                    F.sum(F.col("count") - 1),
                    F.lit(0),
                ).alias("duplicate_count")
            )
            .first()["duplicate_count"]
        )
    else:
        duplicate_grain_count = 0

    silver_metric_rows.append(
        (
            "silver",
            table_name,
            row_count,
            null_business_key_count,
            int(duplicate_grain_count),
        )
    )

print("Silver metric rows created:", len(silver_metric_rows))

In [0]:
silver_quality_metrics_df = spark.createDataFrame(
    silver_metric_rows,
    schema=[
        "layer",
        "table_name",
        "row_count",
        "null_business_key_count",
        "duplicate_grain_count",
    ],
)

silver_quality_metrics_df = (
    silver_quality_metrics_df
    .withColumn(
        "quality_status",
        F.when(
            (F.col("row_count") > 0)
            & (F.col("null_business_key_count") == 0)
            & (F.col("duplicate_grain_count") == 0),
            F.lit("PASS"),
        ).otherwise(F.lit("REVIEW")),
    )
    .withColumn(
        "_quality_reported_at",
        F.current_timestamp(),
    )
    .orderBy("table_name")
)

display(silver_quality_metrics_df)

In [0]:
GOLD_BASE_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist"
)

GOLD_TABLE_CONFIG = {
    "dim_customers": {
        "business_keys": ["customer_id"],
        "grain": ["customer_id"],
    },
    "dim_sellers": {
        "business_keys": ["seller_id"],
        "grain": ["seller_id"],
    },
    "dim_products": {
        "business_keys": ["product_id"],
        "grain": ["product_id"],
    },
    "dim_dates": {
        "business_keys": ["date_key"],
        "grain": ["date_key"],
    },
    "fact_orders": {
        "business_keys": ["order_id"],
        "grain": ["order_id"],
    },
    "fact_order_items": {
        "business_keys": ["order_id", "order_item_id"],
        "grain": ["order_id", "order_item_id"],
    },
    "fact_payments": {
        "business_keys": ["order_id", "payment_sequential"],
        "grain": ["order_id", "payment_sequential"],
    },
    "fact_reviews": {
    "business_keys": ["review_id", "order_id"],
    "grain": ["review_id", "order_id"],
    },
}

In [0]:
gold_tables = {}

for table_name in GOLD_TABLE_CONFIG:
    gold_tables[table_name] = (
        spark.read
        .format("delta")
        .load(f"{GOLD_BASE_PATH}/{table_name}")
    )

print(f"Loaded {len(gold_tables)} Gold tables.")

In [0]:
for table_name, config in GOLD_TABLE_CONFIG.items():
    table_columns = set(gold_tables[table_name].columns)

    required_columns = set(
        config["business_keys"] + config["grain"]
    )

    missing_columns = required_columns - table_columns

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

print("Gold metric configuration validation passed.")

In [0]:
gold_metric_rows = []

for table_name, config in GOLD_TABLE_CONFIG.items():
    table_df = gold_tables[table_name]

    row_count = table_df.count()

    null_key_condition = None

    for key_column in config["business_keys"]:
        current_condition = F.col(key_column).isNull()

        null_key_condition = (
            current_condition
            if null_key_condition is None
            else null_key_condition | current_condition
        )

    null_business_key_count = (
        table_df
        .filter(null_key_condition)
        .count()
    )

    duplicate_grain_count = (
        table_df
        .groupBy(*config["grain"])
        .count()
        .filter(F.col("count") > 1)
        .agg(
            F.coalesce(
                F.sum(F.col("count") - 1),
                F.lit(0),
            ).alias("duplicate_count")
        )
        .first()["duplicate_count"]
    )

    gold_metric_rows.append(
        (
            "gold",
            table_name,
            row_count,
            null_business_key_count,
            int(duplicate_grain_count),
        )
    )

In [0]:
gold_quality_metrics_df = spark.createDataFrame(
    gold_metric_rows,
    schema=[
        "layer",
        "table_name",
        "row_count",
        "null_business_key_count",
        "duplicate_grain_count",
    ],
)

gold_quality_metrics_df = (
    gold_quality_metrics_df
    .withColumn(
        "quality_status",
        F.when(
            (F.col("row_count") > 0)
            & (F.col("null_business_key_count") == 0)
            & (F.col("duplicate_grain_count") == 0),
            F.lit("PASS"),
        ).otherwise(F.lit("REVIEW")),
    )
    .withColumn(
        "_quality_reported_at",
        F.current_timestamp(),
    )
    .orderBy("table_name")
)

display(gold_quality_metrics_df)

In [0]:
data_quality_metrics_df = (
    silver_quality_metrics_df
    .unionByName(gold_quality_metrics_df)
    .orderBy("layer", "table_name")
)

display(data_quality_metrics_df)

In [0]:
DATA_QUALITY_METRICS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/data_quality/data_quality_metrics"
)

In [0]:
(
    data_quality_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DATA_QUALITY_METRICS_PATH)
)

In [0]:
data_quality_metrics_df = (
    silver_quality_metrics_df
    .unionByName(gold_quality_metrics_df)
    .orderBy("layer", "table_name")
)

print("Combined metric rows:", data_quality_metrics_df.count())
display(data_quality_metrics_df)